In [17]:
# imports -------------------------------------------------------------------------#
import os
import argparse
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms  # For data augmentation
from torchinfo import summary
from ema import EMA
from datasets import MnistDataset
from transforms import RandomRotation
from models.modelM3 import ModelM3
from models.modelM5 import ModelM5
from models.modelM7 import ModelM7

In [18]:
p = argparse.ArgumentParser()
p.add_argument("--gpu", default=0, type=int)
args, unknown = p.parse_known_args()

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)

In [19]:
def run(seed=0, epochs=150, kernel_size=5, logdir="temp"):
    # random number generator seed ------------------------------------------------#
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    # file names ------------------------------------------------------------------#
    if not os.path.exists("logs/%s" % logdir):
        os.makedirs("logs/%s" % logdir)
    OUTPUT_FILE = str("logs/%s/logM%s-%03d.out" % (logdir, kernel_size, seed))
    MODEL_FILE = str("logs/%s/modelM%s-%03d.pth" % (logdir, kernel_size, seed))

    # enable GPU usage ------------------------------------------------------------#
    use_cuda = torch.cuda.is_available()
    print("Is CUDA available?: ", use_cuda)

    if not use_cuda:
        print("WARNING: CPU would be used for training.")
        exit(0)
    else:
        device = torch.device("cuda" if use_cuda else "cpu")
        print(f"Using device: {device}")
        print("Number of GPUs available: ", torch.cuda.device_count())
        print("GPUs properties: ", torch.cuda.get_device_properties(0))

    # data augmentation methods ---------------------------------------------------#
    transform = transforms.Compose(
        [
            RandomRotation(20, seed=seed),
            transforms.RandomAffine(0, translate=(0.2, 0.2)),
        ]
    )

    # data loader -----------------------------------------------------------------#
    train_dataset = MnistDataset(training=True, transform=transform)
    test_dataset = MnistDataset(training=False, transform=None)
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=120, shuffle=True
    )
    test_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=100, shuffle=False
    )

    # model selection -------------------------------------------------------------#
    if kernel_size == 3:
        model = ModelM3().to(device)
    elif kernel_size == 5:
        model = ModelM5().to(device)
    elif kernel_size == 7:
        model = ModelM7().to(device)

    summary(model, (1, 1, 28, 28))

    # hyperparameter selection ----------------------------------------------------#
    ema = EMA(model, decay=0.999)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)

    # delete result file ----------------------------------------------------------#
    f = open(OUTPUT_FILE, "w")
    f.close()

    # global variables ------------------------------------------------------------#
    g_step = 0
    max_correct = 0

    # training and evaluation loop ------------------------------------------------#
    for epoch in range(epochs):
        # --------------------------------------------------------------------------#
        # train process                                                            #
        # --------------------------------------------------------------------------#
        model.train()
        train_loss = 0
        train_corr = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device, dtype=torch.int64)
            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            train_pred = output.argmax(dim=1, keepdim=True)
            train_corr += train_pred.eq(target.view_as(train_pred)).sum().item()
            train_loss += F.nll_loss(output, target, reduction="sum").item()
            loss.backward()
            optimizer.step()
            g_step += 1
            ema(model, g_step)
            if batch_idx % 100 == 0:
                print(
                    "Model: M{}\tTrain Epoch: {} [{:05d}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                        kernel_size,
                        epoch,
                        batch_idx * len(data),
                        len(train_loader.dataset),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )
        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * train_corr / len(train_loader.dataset)

        # --------------------------------------------------------------------------#
        # test process                                                             #
        # --------------------------------------------------------------------------#
        model.eval()
        ema.assign(model)
        test_loss = 0
        correct = 0
        total_pred = np.zeros(0)
        total_target = np.zeros(0)
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device, dtype=torch.int64)
                output = model(data)
                test_loss += F.nll_loss(output, target, reduction="sum").item()
                pred = output.argmax(dim=1, keepdim=True)
                total_pred = np.append(total_pred, pred.cpu().numpy())
                total_target = np.append(total_target, target.cpu().numpy())
                correct += pred.eq(target.view_as(pred)).sum().item()
            if max_correct < correct:
                torch.save(model.state_dict(), MODEL_FILE)
                max_correct = correct
                print("Best accuracy! correct images: %5d" % correct)
        ema.resume(model)

        # --------------------------------------------------------------------------#
        # output                                                                   #
        # --------------------------------------------------------------------------#
        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct / len(test_loader.dataset)
        best_test_accuracy = 100 * max_correct / len(test_loader.dataset)
        print(
            "\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%) (best: {:.2f}%)\n".format(
                test_loss,
                correct,
                len(test_loader.dataset),
                test_accuracy,
                best_test_accuracy,
            )
        )

        f = open(OUTPUT_FILE, "a")
        f.write(
            " %3d %12.6f %9.3f %12.6f %9.3f %9.3f\n"
            % (
                epoch,
                train_loss,
                train_accuracy,
                test_loss,
                test_accuracy,
                best_test_accuracy,
            )
        )
        f.close()

        # --------------------------------------------------------------------------#
        # update learning rate scheduler                                           #
        # --------------------------------------------------------------------------#
        lr_scheduler.step()

In [ ]:
seed = 0
epochs = 5
gpu = 0
logdir = "temp"
trials = 2

for i in range(trials):
    print("Current trial: ", i)
    run(
        seed=seed + i,
        epochs=epochs,
        kernel_size=3,
        logdir=logdir,
    )

    run(
        seed=seed + i,
        epochs=epochs,
        kernel_size=5,
        logdir=logdir,
    )

    run(
        seed=seed + i,
        epochs=epochs,
        kernel_size=7,
        logdir=logdir,
    )

print('Finished training models.')